# Days 2–3 — 2D Gaussians for inverse scattering

**A. Direct fit**: N 2D Gaussians learn a cell image directly (does the representation work?).
**B. Physics fit**: Gaussians learn the cell **only from the scattered-field measurements** (the inverse problem),
with 360 / 180 / 60 receivers, compared against a plain **pixel** baseline (same loss, no Gaussians).

This notebook is self-contained (no GitHub needed). It reuses the data zips already in `MyDrive/thesis/data/`.
`Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`. Takes roughly 10–20 min on a T4.

In [ ]:
NOTEBOOK_VERSION = "day2-2026-10-16"
DRIVE_ROOT = "/content/drive/MyDrive/thesis"
ITERS      = 2000              # optimisation steps per fit
N_DIRECT   = 5                 # images for experiment A
N_PHYSICS  = 3                 # images for experiment B
RECEIVERS  = [360, 180, 60]
print("Notebook version:", NOTEBOOK_VERSION)

## 1. Setup (GPU, Drive, InverseBench, data)

In [ ]:
import os, sys, glob, shutil, zipfile, subprocess
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU (slow! switch runtime to T4 GPU)")

from google.colab import drive
drive.mount("/content/drive")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lmdb", "scikit-image"], check=True)
IB = "/content/InverseBench"
if not os.path.isdir(IB):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/devzhk/InverseBench.git", IB], check=True)

LOCAL_DATA = "/content/data"
for name in ["inv-scatter-test"]:
    zip_path, target = f"{DRIVE_ROOT}/data/{name}.zip", f"{LOCAL_DATA}/{name}"
    if not os.path.exists(zip_path):
        raise RuntimeError(f"{zip_path} not found - run the Day 1 notebook first.") from None
    if not os.path.exists(f"{target}/data.mdb"):
        tmp = f"/content/_unzip_{name}"
        shutil.rmtree(tmp, ignore_errors=True)
        zipfile.ZipFile(zip_path).extractall(tmp)
        os.makedirs(LOCAL_DATA, exist_ok=True)
        shutil.move(os.path.dirname(glob.glob(f"{tmp}/**/data.mdb", recursive=True)[0]), target)
OUT = f"{DRIVE_ROOT}/outputs/day2"
os.makedirs(OUT, exist_ok=True)
print("Setup done. Results will be saved to", OUT)

## 2. The Gaussian model and fitting code
(Also kept in the repo as `day2/gs2d.py`.)

In [ ]:
%%writefile /content/gs2d.py
"""2D Gaussian representation of an image + fitting (direct or through the scattering physics).

An image f(x) on a 128x128 grid is represented as a sum of N anisotropic 2D Gaussians:
    f(x) = sum_i  a_i * exp(-0.5 * (x - mu_i)^T  Sigma_i^{-1}  (x - mu_i))
    Sigma_i = R(theta_i) diag(s_i)^2 R(theta_i)^T
Parameters per Gaussian (stored as one (N, 6) tensor -> easy to feed to Mamba later):
    [mu_x, mu_y, log_sx, log_sy, theta, a_raw],  a = softplus(a_raw) >= 0 (permittivity is non-negative)
Coordinates are in pixels (0..127).
"""
import math
import time

import numpy as np
import torch
import torch.nn.functional as F


# ----------------------------------------------------------------------------- model
class Gaussians2D(torch.nn.Module):
    def __init__(self, params):
        super().__init__()
        self.params = torch.nn.Parameter(params)

    @property
    def n(self):
        return self.params.shape[0]

    @staticmethod
    def init_from_image(img, n, init_scale=2.0, generator=None):
        """Place n Gaussians at pixels sampled proportionally to intensity (+ some uniform)."""
        H, W = img.shape
        w = img.flatten().clamp(min=0) + 0.05 * img.max().clamp(min=1e-6)
        idx = torch.multinomial(w / w.sum(), n, replacement=True, generator=generator)
        ys, xs = (idx // W).float(), (idx % W).float()
        jitter = torch.rand(n, 2, generator=generator, device=img.device) - 0.5
        amp = img.flatten()[idx].clamp(min=0.02)
        return Gaussians2D._pack(xs + jitter[:, 0], ys + jitter[:, 1], init_scale, amp)

    @staticmethod
    def init_uniform(n, H, W, init_scale=3.0, amp=0.05, device="cpu", generator=None):
        """Random placement, no knowledge of the image (used for the physics fit)."""
        xs = torch.rand(n, generator=generator, device=device) * (W - 1)
        ys = torch.rand(n, generator=generator, device=device) * (H - 1)
        return Gaussians2D._pack(xs, ys, init_scale, torch.full((n,), amp, device=device))

    @staticmethod
    def _pack(xs, ys, scale, amp):
        n = xs.shape[0]
        log_s = torch.full((n,), math.log(scale), device=xs.device)
        theta = torch.zeros(n, device=xs.device)
        a_raw = torch.log(torch.expm1(amp))  # inverse softplus
        return torch.stack([xs, ys, log_s, log_s, theta, a_raw], dim=1)

    def render(self, H, W, chunk=512):
        p = self.params
        mu, log_s, theta = p[:, 0:2], p[:, 2:4].clamp(-2.0, 4.0), p[:, 4]
        a = F.softplus(p[:, 5])
        s2 = torch.exp(2 * log_s)
        c, s = torch.cos(theta), torch.sin(theta)
        # inverse covariance (conic) entries
        A = c * c / s2[:, 0] + s * s / s2[:, 1]
        B = c * s * (1 / s2[:, 0] - 1 / s2[:, 1])
        C = s * s / s2[:, 0] + c * c / s2[:, 1]
        yy, xx = torch.meshgrid(torch.arange(H, device=p.device, dtype=p.dtype),
                                torch.arange(W, device=p.device, dtype=p.dtype), indexing="ij")
        xx, yy = xx.flatten(), yy.flatten()
        out = torch.zeros(H * W, device=p.device, dtype=p.dtype)
        for i in range(0, self.n, chunk):  # chunk over Gaussians to bound memory
            dx = xx[None] - mu[i:i + chunk, 0:1]
            dy = yy[None] - mu[i:i + chunk, 1:2]
            q = A[i:i + chunk, None] * dx * dx + 2 * B[i:i + chunk, None] * dx * dy + C[i:i + chunk, None] * dy * dy
            out = out + (a[i:i + chunk, None] * torch.exp(-0.5 * q)).sum(0)
        return out.view(H, W)


# ----------------------------------------------------------------------------- physics
class FastScatter:
    """Same linear operator as InverseBench's InverseScatter.forward(f, unnormalize=False),
    rewritten as two small matmuls in complex64 so it is fast on a GPU:
        y[t, r] = dx*dy * sum_p f_p * conj(uinc[p, t]) * G[p, r]
    """

    def __init__(self, ib_operator, device):
        self.dxdy = ib_operator.dx * ib_operator.dy
        uinc = ib_operator.uinc_dom_set[..., 0].reshape(-1, ib_operator.numTrans)       # (P, T)
        G = ib_operator.sensor_greens_function_set[..., 0].reshape(-1, ib_operator.numRec)  # (P, R)
        self.uinc_conj = torch.conj(uinc).to(device=device, dtype=torch.complex64)
        self.G = G.to(device=device, dtype=torch.complex64)

    def __call__(self, f):  # f: (H, W) real
        src = f.reshape(-1, 1).to(torch.complex64) * self.uinc_conj   # (P, T)
        return self.dxdy * (src.T @ self.G)                            # (T, R)


# ----------------------------------------------------------------------------- metrics
def psnr(x, gt):
    mse = torch.mean((x.clamp(0, 1) - gt) ** 2).item()
    return 10 * math.log10(1.0 / max(mse, 1e-12))


def ssim(x, gt):
    from skimage.metrics import structural_similarity
    return float(structural_similarity(x.clamp(0, 1).detach().cpu().numpy(), gt.cpu().numpy(), data_range=1.0))


def rel_l2(x, gt):
    return (torch.linalg.norm(x - gt) / torch.linalg.norm(gt)).item()


# ----------------------------------------------------------------------------- fitting
def _densify_prune(model, opt, residual, n_add, n_max, amp_min=1e-3, init_scale=1.5):
    """Error-guided densification: add Gaussians where |residual| is largest; prune near-zero ones."""
    p = model.params.data
    keep = F.softplus(p[:, 5]) > amp_min
    new = []
    if n_add > 0 and p.shape[0] < n_max:
        H, W = residual.shape
        k = min(n_add, n_max - int(keep.sum()))
        if k > 0:
            idx = torch.topk(residual.abs().flatten(), k).indices
            ys, xs = (idx // W).float(), (idx % W).float()
            amp = residual.flatten()[idx].abs().clamp(min=0.02)
            new = [Gaussians2D._pack(xs, ys, init_scale, amp).to(p)]
    params = torch.cat([p[keep]] + new, 0)
    model.params = torch.nn.Parameter(params)
    lr = opt.param_groups[0]["lr"]
    return torch.optim.Adam([model.params], lr=lr)


def fit(gt, mode="direct", op=None, y=None, n_init=200, n_max=600, iters=2000, lr=0.05,
        densify_every=250, densify_until=0.6, n_add=50, init="image", seed=0, log_every=500, verbose=True):
    """Fit Gaussians to one image.

    mode="direct"  : loss = MSE(render, gt)                       (sanity check)
    mode="physics" : loss = ||op(render) - y||^2 / ||y||^2         (inverse problem, gt only used for metrics)
    """
    device = gt.device
    H, W = gt.shape
    g = torch.Generator(device=device).manual_seed(seed)
    if init == "image":
        params = Gaussians2D.init_from_image(gt, n_init, generator=g)
    else:
        params = Gaussians2D.init_uniform(n_init, H, W, device=device, generator=g)
    model = Gaussians2D(params)
    opt = torch.optim.Adam([model.params], lr=lr)
    ynorm = (y.abs() ** 2).sum() if y is not None else None
    t0 = time.time()
    hist = []
    for it in range(1, iters + 1):
        img = model.render(H, W)
        if mode == "direct":
            loss = F.mse_loss(img, gt)
        else:
            loss = ((op(img) - y).abs() ** 2).sum() / ynorm
        opt.zero_grad()
        loss.backward()
        opt.step()
        # cosine learning-rate decay
        for pg in opt.param_groups:
            pg["lr"] = lr * (0.05 + 0.95 * 0.5 * (1 + math.cos(math.pi * it / iters)))
        if densify_every and it % densify_every == 0 and it < densify_until * iters:
            with torch.no_grad():
                if mode == "direct":
                    residual = gt - model.render(H, W)
                else:  # back-project the measurement residual (adjoint) -> where the image is wrong
                    r = (y - op(model.render(H, W))).to(torch.complex64)
                    residual = (op.dxdy * (op.uinc_conj.conj() * (op.G.conj() @ r.T))).real.sum(1).view(H, W)
                    residual = residual / residual.abs().max().clamp(min=1e-12) * 0.3
            opt = _densify_prune(model, opt, residual, n_add, n_max)
        if verbose and (it % log_every == 0 or it == iters):
            with torch.no_grad():
                img = model.render(H, W)
            hist.append((it, loss.item(), psnr(img, gt)))
            print(f"  iter {it:5d}  loss {loss.item():.3e}  PSNR {psnr(img, gt):6.2f} dB  #G {model.n}")
    with torch.no_grad():
        img = model.render(H, W)
    return model, img.detach(), {"psnr": psnr(img, gt), "ssim": ssim(img, gt), "rel_l2": rel_l2(img, gt),
                                 "n_gauss": model.n, "time_s": time.time() - t0}


def fit_pixels(gt, op, y, iters=2000, lr=0.01):
    """Baseline: optimise the 128x128 pixels directly (no Gaussians), same physics loss, f >= 0."""
    H, W = gt.shape
    x = torch.zeros(H, W, device=gt.device, requires_grad=True)
    opt = torch.optim.Adam([x], lr=lr)
    ynorm = (y.abs() ** 2).sum()
    t0 = time.time()
    for it in range(iters):
        loss = ((op(x) - y).abs() ** 2).sum() / ynorm
        opt.zero_grad()
        loss.backward()
        opt.step()
        with torch.no_grad():
            x.clamp_(min=0.0)  # projection onto f >= 0
    img = x.detach()
    return img, {"psnr": psnr(img, gt), "ssim": ssim(img, gt), "rel_l2": rel_l2(img, gt),
                 "n_gauss": 0, "time_s": time.time() - t0}

In [ ]:
import importlib, numpy as np, lmdb, pandas as pd, matplotlib.pyplot as plt
sys.path.insert(0, "/content"); sys.path.insert(0, IB)
import gs2d; importlib.reload(gs2d)
from gs2d import fit, fit_pixels, FastScatter
from inverse_problems.inverse_scatter import InverseScatter

env = lmdb.open(f"{LOCAL_DATA}/inv-scatter-test", readonly=True, lock=False)
with env.begin() as txn:
    n_img = max(N_DIRECT, N_PHYSICS)
    imgs = [np.frombuffer(txn.get(str(i).encode()), dtype=np.float32).reshape(128, 128).copy() for i in range(n_img)]
gts = [torch.from_numpy(im).to(DEVICE) for im in imgs]
print(f"Loaded {len(gts)} test images")

## 3. Experiment A — direct fit (sanity check)

In [ ]:
rows_a, recon_a = [], []
for i in range(N_DIRECT):
    print(f"Image {i}")
    _, img, r = fit(gts[i], mode="direct", iters=ITERS, init="image", log_every=ITERS // 2)
    rows_a.append({"image": i, **r}); recon_a.append(img.cpu().numpy())
df_a = pd.DataFrame(rows_a).round(4)
df_a.to_csv(f"{OUT}/A_direct_fit.csv", index=False)
display(df_a); print("Mean PSNR:", round(df_a.psnr.mean(), 2), "dB   (target > 30 dB)")

fig, ax = plt.subplots(2, N_DIRECT, figsize=(3 * N_DIRECT, 6))
for i in range(N_DIRECT):
    ax[0, i].imshow(imgs[i], cmap="gray", vmin=0, vmax=1); ax[0, i].set_title(f"ground truth {i}")
    ax[1, i].imshow(recon_a[i], cmap="gray", vmin=0, vmax=1)
    ax[1, i].set_title(f"{rows_a[i]['n_gauss']} Gaussians\n{rows_a[i]['psnr']:.1f} dB")
for a in ax.flat: a.axis("off")
plt.tight_layout(); plt.savefig(f"{OUT}/A_direct_fit.png", dpi=120); plt.show()

## 4. Experiment B — reconstruction from measurements only (the inverse problem)

In [ ]:
rows_b, examples = [], {}
for R in RECEIVERS:
    ib = InverseScatter(numRec=R, numTrans=20, svd=False, device=DEVICE)   # same settings as the benchmark
    op = FastScatter(ib, DEVICE)
    # check our fast operator against InverseBench's own forward()
    y_ib = ib.forward(gts[0][None, None].double(), unnormalize=False)[0]
    err = ((op(gts[0]) - y_ib.to(torch.complex64)).abs().max() / y_ib.abs().max()).item()
    print(f"\n===== {R} receivers  (operator check: rel. error {err:.1e}) =====")
    for i in range(N_PHYSICS):
        torch.manual_seed(i)
        y = op(gts[i]); y = y + 1e-4 * torch.randn_like(y)        # benchmark noise level
        print(f"Image {i} - Gaussians")
        _, img_g, rg = fit(gts[i], mode="physics", op=op, y=y, init="uniform", iters=ITERS, log_every=ITERS // 2)
        img_p, rp = fit_pixels(gts[i], op, y, iters=ITERS)
        print(f"Image {i} - pixels: PSNR {rp['psnr']:.2f} dB")
        rows_b += [{"receivers": R, "image": i, "method": "Gaussians", **rg},
                   {"receivers": R, "image": i, "method": "pixels", **rp}]
        if i == 0: examples[R] = (img_g.cpu().numpy(), img_p.cpu().numpy(), rg, rp)
df_b = pd.DataFrame(rows_b).round(4)
df_b.to_csv(f"{OUT}/B_physics_fit.csv", index=False)
summary = df_b.groupby(["receivers", "method"])[["psnr", "ssim", "rel_l2", "n_gauss", "time_s"]].mean().round(3)
summary.to_csv(f"{OUT}/B_summary.csv")
display(summary)

In [ ]:
fig, ax = plt.subplots(len(RECEIVERS), 3, figsize=(9, 3 * len(RECEIVERS)))
for k, R in enumerate(RECEIVERS):
    g, p, rg, rp = examples[R]
    ax[k, 0].imshow(imgs[0], cmap="gray", vmin=0, vmax=1); ax[k, 0].set_title("ground truth")
    ax[k, 1].imshow(g, cmap="gray", vmin=0, vmax=1); ax[k, 1].set_title(f"Gaussians, {R} rec.\n{rg['psnr']:.1f} dB / SSIM {rg['ssim']:.2f}")
    ax[k, 2].imshow(p, cmap="gray", vmin=0, vmax=1); ax[k, 2].set_title(f"pixels, {R} rec.\n{rp['psnr']:.1f} dB / SSIM {rp['ssim']:.2f}")
for a in ax.flat: a.axis("off")
plt.tight_layout(); plt.savefig(f"{OUT}/B_physics_fit.png", dpi=120); plt.show()
print("All results saved in", OUT)

## Done ✅
Send Claude: the two tables (`df_a` mean PSNR and the `summary` table) and the two figures
(also saved in `MyDrive/thesis/outputs/day2/`).